# Transformer Architectures — Completed Hands-On Lab

**Source:** `AI_lab_transformers.ipynb` and `Transformer.pdf`

This solution demonstrates the three major Transformer patterns: encoder–decoder translation, decoder-only generation, and encoder-only classification. Models run on CPU and are released after each example to keep memory use manageable. The first run downloads the pretrained weights; later runs use the local Hugging Face cache.

In [1]:
# Run this cell once only if the packages are missing.
# %pip install -q "transformers>=4.45,<5" sentencepiece accelerate

import gc
import os
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

torch.manual_seed(42)
print("PyTorch:", torch.__version__)

/home/kimi/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.8.0+cu128


## 1. Encoder–decoder architecture: English → German

The encoder builds contextual representations of the complete English input. At each output step, the decoder uses causal self-attention over previous German tokens and **cross-attention** over the encoder outputs. This matches the original sequence-to-sequence purpose of the Transformer.

In [2]:
translation_model = "Helsinki-NLP/opus-mt-en-de"
translator = pipeline("translation", model=translation_model, device=-1)

english_sentences = [
    "Plants create energy through a process known as photosynthesis.",
    "I love machine learning."
]
german = translator(english_sentences, max_new_tokens=60)
for source, result in zip(english_sentences, german):
    print("English:", source)
    print("German :", result["translation_text"])
    print()

del translator
gc.collect()

/home/kimi/.local/lib/python3.12/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use cpu


English: Plants create energy through a process known as photosynthesis.
German : Pflanzen erzeugen Energie durch einen Prozess, der als Photosynthese bekannt ist.

English: I love machine learning.
German : Ich liebe maschinelles Lernen.



127

## 2. Encoder–decoder architecture: English → French with T5

T5 frames tasks as text-to-text problems. The prefix `translate English to French:` tells the model which task to perform.

In [3]:
t5_name = "google-t5/t5-small"
t5_tokenizer = AutoTokenizer.from_pretrained(t5_name)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_name)

text = "translate English to French: Let's go have some wine, because that is how we shine."
inputs = t5_tokenizer(text, return_tensors="pt")
with torch.no_grad():
    output_ids = t5_model.generate(**inputs, max_new_tokens=50)
french = t5_tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("French:", french)

del t5_model, t5_tokenizer, inputs, output_ids
gc.collect()

French: Voulons-nous avoir un vin, car c'est ainsi que nous en éclairons.


117

## 3. Encoder–decoder architecture: English → Hinglish

This completes the additional translation example from the original notebook. Hinglish mixes Hindi and English; the specialised Flan-T5 checkpoint has been fine-tuned for that conversion.

In [4]:
hinglish_name = "rvv-karma/English2Hinglish-Flan-T5-Base"
hinglish = pipeline("text2text-generation", model=hinglish_name, device=-1)
input_text = "life is crazy. Not me"
result = hinglish(input_text, max_new_tokens=50)
print("English :", input_text)
print("Hinglish:", result[0]["generated_text"])

del hinglish
gc.collect()

Device set to use cpu


English : life is crazy. Not me
Hinglish: Life crazy haiNot me


0

## 4. Decoder-only architecture: GPT-style text generation

A decoder-only language model learns \(P(x_t\mid x_{<t})\). During generation it predicts one token, appends it to the prefix, and repeats. Causal self-attention prevents access to future tokens; there is no separate encoder or cross-attention.

In [5]:
generator = pipeline("text-generation", model="gpt2", device=-1)
prompt = "The future of AI is"
outputs = generator(
    prompt,
    max_new_tokens=30,
    num_return_sequences=2,
    do_sample=True,
    temperature=0.8,
    top_p=0.95,
    pad_token_id=generator.tokenizer.eos_token_id,
)
for i, output in enumerate(outputs, 1):
    print(f"Generation {i}: {output['generated_text']}\n")

del generator
gc.collect()

Device set to use cpu


Generation 1: The future of AI is at stake."

This article was originally published on The Conversation. Read the original article.

Generation 2: The future of AI is likely to be based on the very next step in the development of artificial intelligence," said Mark Anderson, chief executive of research firm DeepMind, adding that



0

## 5. Encoder-only architecture: sentiment analysis

An encoder-only model processes all input tokens bidirectionally. A classification head maps the resulting sequence representation to sentiment labels; the model does not autoregressively generate an answer.

In [6]:
classifier = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=-1,
)
texts = [
    "I love how clearly this lab explains attention.",
    "I'd want to kick you and ensure that you are hurt."
]
results = classifier(texts)
for text, result in zip(texts, results):
    print(text)
    print(f"  label={result['label']}, confidence={result['score']:.4f}")

del classifier
gc.collect()

Device set to use cpu


I love how clearly this lab explains attention.
  label=POSITIVE, confidence=0.9998
I'd want to kick you and ensure that you are hurt.
  label=NEGATIVE, confidence=0.9734


33

## 6. Conceptual answers and comparison

**What was the Transformer initially introduced for?** Machine translation, and more generally sequence-to-sequence learning.

| Architecture | Attention pattern | Typical task demonstrated |
|---|---|---|
| Encoder–decoder | Bidirectional encoder self-attention; causal decoder self-attention; encoder–decoder cross-attention | Translation |
| Decoder-only | Causal self-attention | Next-token text generation |
| Encoder-only | Bidirectional self-attention | Sentiment classification |

**Important mechanisms:** attention computes weighted combinations of value vectors using query–key compatibility; self-attention uses projections of the same sequence; cross-attention uses decoder queries and encoder keys/values; multi-head attention learns several projections in parallel; positional information supplies token order; causal masking blocks future positions.

All requested hands-on tasks are complete: machine translation, GPT-type generation, encoder-based sentiment analysis, and architecture comparison.